In [10]:
import re
import math
import yaml
from pathlib import Path 
import decord
import pandas as pd
import numpy as np
import torch
import cv2
import hickle as hkl
import logging
from scipy.spatial.transform import Rotation as R
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view


# from spinflow.util.math_utils import (odom_to_local_pose)
# from scripts.mapping.compute_odom import (interpolate_se3)

_rx = re.compile(
    r"""^ride_
        (?P<ride_dir>\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2})_
        (?P<robot_name>[^_]+)_
        (?P<drive_dir>\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}_\d+)_
        seq_
        (?P<start_frame>\d+)_
        (?P<end_frame>\d+)""",
    re.VERBOSE,
)

In [11]:
def parse_ride_string(s: str) -> dict:
    """
    Extract ride_dir, robot_name, drive_dir, start_frame and end_frame
    from strings such as:
        ride_2025-07-02-16-33-00_ferrite3_2025-07-02-16-39-39_0_seq_252_332
    """
    m = _rx.match(s)
    if not m:
        raise ValueError(f"bad ride string: {s}")

    d = m.groupdict()
    d["start_frame"] = int(d["start_frame"])
    d["end_frame"]   = int(d["end_frame"])
    return d

def draw_trajectory_on_image(image, trajectory, color=(255, 255, 51), thickness=2):
    """
    Draws a trajectory on an image.

    Args:
        image (torch.Tensor or np.ndarray): [B, C, H, W] or [H, W, C] image tensor/array.
        trajectory (torch.Tensor or np.ndarray): [B, N, 2] or [N, 2] trajectory pixel coords.
        color (tuple): BGR color of the trajectory.
        thickness (int): Line thickness (and circle “size”).

    Returns:
        np.ndarray: Annotated image of shape (H, W, 3).
    """
    # --- prepare the image ---
    if isinstance(image, torch.Tensor):
        img_np = image.cpu().numpy()
    else:
        img_np = image.copy()

    # handle batch dimension
    if img_np.ndim == 4:
        # [B, C, H, W] → take first
        img_np = img_np[0]

    # now either [C, H, W] or [H, W, C]
    if img_np.ndim == 3 and img_np.shape[0] in (1, 3):
        # channels-first → channels-last
        img_np = np.transpose(img_np, (1, 2, 0))

    # Normalize image to 0-255 range if needed
    if img_np.dtype != np.uint8:
        img_np = cv2.normalize(img_np, None, 0, 255, cv2.NORM_MINMAX)

    # ensure we have 3-channel BGR
    if img_np.ndim == 2:
        img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)

    annotated = img_np.copy()

    # --- prepare the trajectory ---
    if isinstance(trajectory, torch.Tensor):
        traj = trajectory.cpu().numpy()
    else:
        traj = trajectory

    # handle batch
    if traj.ndim == 3:
        traj = traj[0]

    # cast to int pixel coords
    pts = np.round(traj).astype(int)

    # --- draw circles at each point ---
    # radius = max(1, thickness)
    # for (x, y) in pts:
    #     cv2.circle(annotated, (x, y), radius=radius, color=color, thickness=thickness)

    # --- draw lines connecting them ---
    for p0, p1 in zip(pts[:-1], pts[1:]):
        x0, y0 = int(p0[0]), int(p0[1])
        x1, y1 = int(p1[0]), int(p1[1])
        cv2.line(annotated, (x0, y0), (x1, y1), color=color, thickness=thickness)

    return annotated

def get_pts2pixel_transform(calib_dict):
    """
    Returns a transformation matrix that converts 3D points in LiDAR frame to image pixel coordinates
    Boilerplate function to get the projection matrix from the calibration dictionary

    P =  Pcam @ Eye(Re | 0) @ T_lidar_cam

    Inputs:
        calib_dict: [dict] calibration dictionary
    Outputs:
        pts2pix: [4 x 4] transformation matrix
    """
    T_cam_world = calib_dict['cam_to_world_matrix']
    T_world_cam = np.linalg.inv(T_cam_world)

    T_canon = np.eye(4)
    T_canon[:3, :3] = calib_dict['rectification_matrix']

    M = calib_dict['new_camera_matrix'][:3, :3]
    P_pix_cam = np.eye(4)
    P_pix_cam[:3, :3] = M

    T_world_to_rect = P_pix_cam @ T_canon @ T_world_cam

    return T_world_to_rect


def draw_xyz_on_image(
    image, 
    xyz, 
    infos,
    color=(255, 255, 51), 
    thickness=2
):
    """
    Draws odometry xyz points on image
    """
    if isinstance(image, torch.Tensor):
        assert image.ndim == 4, "Image should be of shape [B, C, H, W]"
        image = image[0].permute(1, 2, 0).cpu().numpy()  # [C, H, W] → [H, W, C]
    if isinstance(xyz, torch.Tensor):
        assert xyz.ndim == 3, "XYZ should be of shape [B, T, 3]"
        xyz = xyz[0].cpu().numpy()
    assert xyz.ndim == 2 and xyz.shape[-1] == 3, "XYZ should be of shape [T, 3]."
    assert image.ndim == 3 and image.shape[-1] in (1, 3), "Image should be of shape [H, W, C] or [H, W]."

    uv = project_xyz_to_pixel(xyz, infos['intrinsics'], infos['T_optical_to_base'])
    if uv is None:
        annotated_image = image.copy()
    else:
        annotated_image = draw_trajectory_on_image(
            image, 
            uv, 
            color=color, 
            thickness=thickness
        )
    annotated_image = cv2.normalize(annotated_image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    # import cv2
    # cv2.imwrite("test.jpg", annotated_image)  # Save for debugging
    return annotated_image

def project_xyz_to_pixel(xyz, intrinsics, T_cam_to_world):
    """
    Projects Nx3 xyz points to image camera intrinsics and extrinsics.
    """
    assert xyz.ndim == 2 and xyz.shape[-1] == 3, "XYZ should be of shape [N, 3]."

    # TODO: implement the projection logic
    calib_dict = {
        'cam_to_world_matrix': T_cam_to_world.cpu().numpy(),
        'new_camera_matrix': intrinsics['K'][0].cpu().numpy(),
        'rectification_matrix': intrinsics['R'][0].cpu().numpy(),
    }
    T_world_to_rect = get_pts2pixel_transform(calib_dict)[0]
    print("T_world_to_rect:", T_world_to_rect)
    pts_homogeneous = np.hstack([xyz, np.ones((xyz.shape[0], 1))])  # [N, 4]
    uv_homogeneous = (T_world_to_rect @ pts_homogeneous.T).T  # [N, 4]
    uv = uv_homogeneous[:, :2] / uv_homogeneous[:, 2:3]  # Normalize by z to get pixel coordinates

    # Clip invalid points to be within image bounds and in front of the camera
    image_H, image_W = intrinsics['image_height'][0].item(), intrinsics['image_width'][0].item()
    valid_mask = (uv[:, 0] >= 0) & (uv[:, 0] < image_W) & \
                 (uv[:, 1] >= 0) & (uv[:, 1] < image_H) & \
                 (uv_homogeneous[:, 2] > 0)  # Ensure points are in front of the camera

    if not valid_mask.any():
        return None
    
    uv = uv[valid_mask]  # Keep only valid points
    return uv

try:
    from pymlg import SO3, SE3
except ImportError:
    logging.error("Please install 'pymlg' package to use this script.")
    raise ImportError("Missing 'pymlg' package. Install it with: pip install pymlg")
# ── tiny helpers to “batchify” scalar SE3 ops ───────────────────────────
def se3_inverse_batch(Ts: np.ndarray) -> np.ndarray:         # (B,4,4)
    return np.stack([SE3.inverse(T) for T in Ts], axis=0)

def se3_log_batch(Ts: np.ndarray) -> np.ndarray:             # (B,6)
    return np.stack([SE3.Log(T).reshape(-1) for T in Ts], 0)

def se3_exp_batch(xis: np.ndarray) -> np.ndarray:            # (B,4,4)
    return np.stack([SE3.Exp(xi) for xi in xis], axis=0)

# ── quat+xyz → SE3 ----------------------------------------------------------
def quat_to_se3(q_wxyz: np.ndarray, xyz: np.ndarray) -> np.ndarray:
    q_xyzw = q_wxyz[:, [1, 2, 3, 0]]           # SciPy order x y z w
    mats   = np.zeros((len(q_wxyz), 4, 4))
    mats[:, 3, 3]  = 1.0
    mats[:, :3, :3] = R.from_quat(q_xyzw).as_matrix()
    mats[:, :3,  3] = xyz
    return mats


# ── batch Lie–group interpolation without library broadcasting -------------
def interpolate_se3(
    cam_ts:     np.ndarray,        # (F,)
    odo_ts:     np.ndarray,        # (N,)
    odo_xyz:    np.ndarray,        # (N,3)
    odo_q_wxyz: np.ndarray         # (N,4)  qw qx qy qz
) -> np.ndarray:                   # → (F,8) ts x y z  qw qx qy qz
    # 1) Convert odometry poses to SE3 matrices
    T_odo = quat_to_se3(odo_q_wxyz, odo_xyz)             # (N,4,4)

    print("Odom timestamps:")
    for i in range(5):
        print(f"{odo_ts[i]:.3f}")
    print("Camera timestamps:")
    for i in range(5):
        print(f"{cam_ts[i]:.3f}")

    # 2) Bracket each camera timestamp
    idx0 = np.searchsorted(odo_ts, cam_ts, side="right") - 1
    idx0 = np.clip(idx0, 0, len(odo_ts) - 2)
    idx1 = idx0 + 1
    alpha = ((cam_ts - odo_ts[idx0]) /
             (odo_ts[idx1] - odo_ts[idx0]))[:, None]     # (F,1)

    T0, T1 = T_odo[idx0], T_odo[idx1]                    # (F,4,4)
    # 3) Relative motion & SE3 interpolation
    Delta = se3_inverse_batch(T0) @ T1                   # (F,4,4)
    xi    = se3_log_batch(Delta)                         # (F,6)
    T_d   = se3_exp_batch(alpha * xi)                    # (F,4,4)
    T_cam = T0 @ T_d                                     # (F,4,4)

    # 4) Unpack xyz + quaternion (qw qx qy qz)
    xyz_cam = T_cam[:, :3, 3]
    q_xyzw  = R.from_matrix(T_cam[:, :3, :3]).as_quat()
    q_wxyz  = q_xyzw[:, [3, 0, 1, 2]]
    return np.hstack([cam_ts.reshape(-1, 1), xyz_cam, q_wxyz])                  # (F,7)

def se3_matrix(x: np.ndarray, y: np.ndarray, z: np.ndarray, q_wxyz: np.ndarray) -> np.ndarray:
    """
    Given arrays x, y, z of shape (N,) and quaternion q_wxyz of shape (N, 4),
    return an array of SE(3) mats shape (N,4,4).
    """
    N = x.shape[0]
    Rt = R.from_quat(q_wxyz[:, [1, 2, 3, 0]]).as_matrix()  # Convert to SciPy's (x y z w) order

    T = np.zeros((N, 4, 4), dtype=float)
    T[:, :3, :3] = Rt
    T[:, :3, 3] = np.column_stack([x, y, z])
    T[:, 3, 3] = 1.0

    return T

def odom_to_local_pose(
    odom: np.ndarray,
    origin: int = 0,
    mode: str = "se2",
) -> np.ndarray:
    """
    Convert a global odometry trace to **local coordinates w.r.t. the first pose**.

    Parameters
    ----------
    odom : (N,8) ndarray
        Global poses ordered `[x  y  z  qw  qx  qy  qz]`
        (scalar–first quaternion, right-handed).
    mode : {"se2","se3"}, default="se2"
        • "se2" → return local **(x,y,yaw)** – identical to the old function.  
        • "se3" → return local **(x,y,z,qw,qx,qy,qz)**.

    Returns
    -------
    out :
        - If *mode="se2"*  → shape **(N,3)**  = `[x_local, y_local, yaw_local]`
        - If *mode="se3"*  → shape **(N,7)**  = `[x_local, y_local, z_local, qw, qx, qy, qz]`
    """
    if odom.shape[1] != 8:
        raise ValueError("odom must be (N,8) with [ts, x y z qw qx qy qz]")

    if mode not in {"se2", "se3"}:
        raise ValueError("mode must be either 'se2' or 'se3'")

    odom_ts   = odom[:, 0:1]          # (N,1)
    xyz_world = odom[:, 1:4]          # (N,3)
    q_wxyz    = odom[:, 4:]          # (N,4) scalar-first

    # ------------------------------------------------------------------ #
    #                      3-D  (x,y,z,q)   branch                       #
    # ------------------------------------------------------------------ #
    # Build SE(3) matrices
    Tt_world = se3_matrix(
        xyz_world[:, 0], xyz_world[:, 1], xyz_world[:, 2], q_wxyz
    )                                                # (N,4,4)
    assert origin < len(Tt_world), f"Origin index {origin} out of bounds for {len(Tt_world)} poses."
    T0_inv   = np.linalg.inv(Tt_world[origin])
    Tt_local = T0_inv @ Tt_world                     # (N,4,4)

    # local translation
    xyz_local = Tt_local[:, :3, 3]                   # (N,3)

    # local rotation → quaternion (scalar-first)
    Rt_local  = Tt_local[:, :3, :3]
    q_xyzw_local = R.from_matrix(Rt_local).as_quat() # (N,4) xyzw
    q_wxyz_local = q_xyzw_local[:, [3, 0, 1, 2]]     # back to wxyz

    return np.column_stack([odom_ts, xyz_local, q_wxyz_local])

def plot_odometry_topdown(
    odom: np.ndarray,
    output_path: str,
    *,
    figsize: tuple[float, float] = (6, 6),
    line_color: str = "C0",
    line_width: float = 2.0,
    marker: str = None,
    marker_size: float = 4.0,
    xlabel: str = "X (m)",
    ylabel: str = "Y (m)",
    title: str = "Top-Down Odometry",
    dpi: int = 150
) -> None:
    """
    Plot a top-down (X vs Y) view of a sequence of odometry poses and save to file.
    
    Parameters
    ----------
    odom : array-like, shape (N, ≥2)
        Sequence of poses. The first two columns must be X and Y coordinates.
        Any additional columns (e.g. Z, quaternion) are ignored.
    output_path : str
        Path to write the figure (e.g. "trajectory.png" or "traj.pdf").
    figsize : tuple, default=(6,6)
        Figure size in inches (width, height).
    line_color : str, default="C0"
        Matplotlib line color.
    line_width : float, default=2.0
        Width of the trajectory line.
    marker : str or None, default=None
        Marker style for points (e.g. "o", "^"); if None no markers are drawn.
    marker_size : float, default=4.0
        Size of the markers, if used.
    xlabel, ylabel, title : str
        Labels and title for the plot.
    dpi : int, default=150
        Resolution of the saved figure.
    """
    # Convert to NumPy array
    arr = np.asarray(odom, dtype=float)
    if arr.ndim != 2 or arr.shape[1] < 2:
        raise ValueError("`odom` must be shape (N, ≥2), with X,Y in columns 0,1.")
    
    x = arr[:, 0]
    y = arr[:, 1]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(x, y, color=line_color, linewidth=line_width,
            marker=marker, markersize=marker_size)
    ax.set_aspect("equal", "box")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.5)
    
    # Save and close
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

def quat_to_yaw(qw, qx, qy, qz):
    siny_cosp = 2.0 * (qw * qz + qx * qy)
    cosy_cosp = 1.0 - 2.0 * (qy * qy + qz * qz)
    return math.atan2(siny_cosp, cosy_cosp)

def angular_diff(a, b):
    return (b - a + math.pi) % (2 * math.pi) - math.pi

def window_metrics(odom_np: np.ndarray, horizon_frames: int):
    """
    Vectorised version: returns three 1-D arrays (length N-H) so that
    entry *k* corresponds to the centred window whose **middle frame index**
    is k + H//2.

    Returns
    -------
    dist      : (N-H,)  Euclidean start→end distance  [metres]
    max_dyaw  : (N-H,)  max |yaw - yaw_start|         [radians]
    max_omega : (N-H,)  max |d(yaw)/dt|               [deg/s]
    """
    H       = horizon_frames
    N       = odom_np.shape[0]
    window  = H + 1                                    # samples in each window

    # ---------------- positions --------------------
    pos  = odom_np[:, 1:4]                             # (N,3)
    dist = np.linalg.norm(pos[H:] - pos[:-H], axis=1) # (N-H,)

    # ---------------- yaw series ------------------
    q      = odom_np[:, 4:8]                          # (N,4)
    yaw    = np.array([quat_to_yaw(*row) for row in q])
    yaw_win = sliding_window_view(yaw, window_shape=window)  # (N-H,window)

    yaw0       = yaw_win[:, 0:1]                      # (N-H,1)
    disp_all   = angular_diff(yaw0, yaw_win)          # broadcast → (N-H,window)
    max_dyaw   = np.max(np.abs(disp_all), axis=1)     # (N-H,)

    # ---------------- instantaneous ω -------------
    dy   = angular_diff(yaw_win[:, :-1], yaw_win[:, 1:])     # (N-H,H)
    ts   = odom_np[:, 0]
    dt   = sliding_window_view(ts, window_shape=window)
    dt   = dt[:, 1:] - dt[:, :-1]                     # (N-H,H)
    omega = np.abs(np.degrees(dy) / dt)               # deg/s
    max_omega = np.max(omega, axis=1)                 # (N-H,)

    return dist, max_dyaw, max_omega                 # each length N-H

def blend_mask(
    image: np.ndarray,
    mask: np.ndarray,
    color: tuple = (51, 255, 255),
    alpha: float = 0.5
) -> np.ndarray:
    """
    Blends a mask onto an image with a specified color and transparency.

    Args:
        image:  H×W×3 uint8 RGB image.
        mask:   H×W bool mask array.
        color:  RGB color for the mask overlay.
        alpha:  Transparency level (0–1).

    Returns:
        Blended H×W×3 uint8 image.
    """
    colored_mask = image.copy()
    colored_mask[mask] = color
    blended = cv2.addWeighted(image, 1 - alpha, colored_mask, alpha, 0)
    return blended


def draw_trajectory_on_video(
    video_np,
    video_ts_np,
    odometry_np,
    odometry_ts_np,
    camera_info,
    tracker_info=None,
    enable_cam_offset=False,
    video_path="annotated_video.mp4"
):
    """
    Draws the odometry trajectory on the video frames.
    
    Args:
        video_np (np.ndarray): Video frames as a numpy array of shape [T, H, W, C].
        video_ts_np (np.ndarray): Timestamps for each video frame.
        odometry_np (np.ndarray): Odometry data as a numpy array of shape [T, 3].
        odometry_ts_np (np.ndarray): Timestamps for each odometry data point.
        camera_info (dict): Camera calibration information.
    
    Returns:
        np.ndarray: Annotated video frames with the trajectory drawn.
    """
    writer = cv2.VideoWriter(video_path, 
                            cv2.VideoWriter_fourcc(*'mp4v'), 
                            15,
                            (video_np.shape[2], video_np.shape[1]))
    
    path_mask = tracker_info['path_mask']
    visibility = tracker_info['visibility']
    last_visible_frame = np.argmin(visibility.sum(axis=1) > 20)
    path_overlay_frame = None
    for i, frame in enumerate(video_np):
        #1 Truncate odometry to the current video frame
        current_video_ts = video_ts_np[i:]
        current_odom_ts = odometry_ts_np[i:]
        assert current_video_ts[0] == current_odom_ts[0], "Video and odometry timestamps do not match at the start."
        current_odom = odometry_np[i:]
        current_frame = video_np[i]

        #2 Transform odometry to relative to current frame
        current_odom_rel = odom_to_local_pose(current_odom, mode="se3") 

        if enable_cam_offset:
            # Convert relative odometry to camera center coordinates
            p_cam_base = camera_info['T_optical_to_base'][0, :3, 3]
            p_cam_base_homo = np.concatenate([p_cam_base, [1]])
            T_base_odom = se3_matrix(
                current_odom_rel[:, 1],
                current_odom_rel[:, 2],
                current_odom_rel[:, 3],
                current_odom_rel[:, 4:8]  # qw, qx, qy, qz
            )
            p_cam_odom = T_base_odom @ p_cam_base_homo.T
            p_cam_odom[:, 2] -= 0.576
            p_cam_odom[:, 0] += 0.27 # offset for cam to center of path mask
        else:
            p_cam_odom = current_odom_rel[:, 1:4]
            p_cam_odom[:, 2] -= 0.317

        #Draw the trajectory on the current frame
        annotated_frame = draw_xyz_on_image(
            current_frame, 
            p_cam_odom[:last_visible_frame, :3],  # Use only the XYZ coordinates
            camera_info,
            color=(255, 255, 51), 
            thickness=6
        )

        if path_mask is not None and i == 0:
            path_overlay_frame = blend_mask(annotated_frame, path_mask)
        
        # 2) -- NEW: heading & angular-velocity over next 20 frames ---------------
        horizon = 10
        begin = max(0, i - horizon)
        end = min(i + horizon, len(odometry_np) - 1)
        origin = i - begin

        if end < len(odometry_np) - 1:
            odometry_window_rel = odometry_np[begin: (end + 1)]
            disp, dyaw, omega = window_metrics(odometry_window_rel, horizon)
            disp = disp.mean()
            max_dyaw = dyaw.max()
            max_omega = omega.max()
            txt = f"dist = {disp:.2f}m | max_dyaw = {math.degrees(max_dyaw):5.1f}deg | max_omega = {max_omega:5.1f}deg/s"
            cv2.putText(
                annotated_frame, txt, (12, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2, cv2.LINE_AA
            )
        
        writer.write(annotated_frame)
    
    writer.release()
    return path_overlay_frame

def se3_matrix(x: np.ndarray, y: np.ndarray, z: np.ndarray, q_wxyz: np.ndarray) -> np.ndarray:
    """
    Given arrays x, y, z of shape (N,) and quaternion q_wxyz of shape (N, 4),
    return an array of SE(3) mats shape (N,4,4).
    """
    N = x.shape[0]
    Rt = R.from_quat(q_wxyz[:, [1, 2, 3, 0]]).as_matrix()  # Convert to SciPy's (x y z w) order

    T = np.zeros((N, 4, 4), dtype=float)
    T[:, :3, :3] = Rt
    T[:, :3, 3] = np.column_stack([x, y, z])
    T[:, 3, 3] = 1.0

    return T

def se3_to_odom(odom: np.ndarray) -> np.ndarray:
    """
    Returns
    -------
    out : (N, 8) ndarray
        Odometry format [x, y, z, qw, qx, qy, qz].
    """
    if odom.shape[1:] != (4, 4):
        raise ValueError("odom must be of shape (N, 4, 4)")

    xyz = odom[:, :3, 3]  # Extract translation
    Rt = odom[:, :3, :3]  # Extract rotation matrix
    q_xyzw = R.from_matrix(Rt).as_quat()
    
    return np.column_stack([xyz, q_xyzw[:, [3, 0, 1, 2]]]) 

In [12]:
rides = [
    "ride_2025-08-04-16-54-00_ferrite4_2025-08-04-16-58-26_0_seq_5577_5727"
    # "ride_2025-05-30-12-09-00_ferrite2_2025-05-30-12-12-42_0_seq_13133_13283",
    # "ride_2025-06-12-16-07-00_ferrite3_2025-06-12-16-10-53_0_seq_1951_2101",
    # "ride_2025-06-10-12-53-00_ferrite2_2025-06-10-12-55-29_0_seq_5147_5297",
    # "ride_2025-06-11-13-42-00_ferrite2_2025-06-11-14-18-08_0_seq_10435_10585",
    # "ride_2025-06-12-14-58-00_ferrite5_2025-06-12-15-00-24_0_seq_3098_3248",
    # "ride_2025-06-06-11-56-00_ferrite2_2025-06-06-11-59-52_0_seq_391_541",
    # "ride_2025-06-13-16-01-00_ferrite5_2025-06-13-16-04-03_0_seq_2629_2779",
    # "ride_2025-07-02-12-52-00_ferrite3_2025-07-02-12-54-24_0_seq_3726_3876",
    # "ride_2025-06-15-15-50-00_ferrite3_2025-06-15-15-52-09_0_seq_2300_2450",

]

for ride_path in rides:
    parsed_inputs = parse_ride_string(ride_path)
    print(f"Parsed inputs: {parsed_inputs}")

    PROJECT_DIR = "/home/ubuntu/playground/spinflow"
    ROBOT_NAME = parsed_inputs["robot_name"]
    RIDE_DIR = f'output_rides_{parsed_inputs["ride_dir"]}'
    DRIVE_DIR = f'ride_{ROBOT_NAME}_{parsed_inputs["drive_dir"]}'
    START_FRAME = parsed_inputs["start_frame"]
    END_FRAME = parsed_inputs["end_frame"]

    ROOT_DIR = f"{PROJECT_DIR}/data/fai_spinflow_raw/{RIDE_DIR}/{DRIVE_DIR}"
    PROCESSED_DIR = f"{PROJECT_DIR}/data/fai_spinflow_processed/{RIDE_DIR}/{DRIVE_DIR}/seq_{START_FRAME}"
    odometry_path = Path(ROOT_DIR) / f"odometry_data_{ROBOT_NAME}.csv"
    camera_info_path = Path(ROOT_DIR) / f"front_camera_info_{ROBOT_NAME}.yaml"
    tf_path = Path(ROOT_DIR) / f"tf_static_{ROBOT_NAME}.yaml"
    tracker_path = Path(PROCESSED_DIR) / "path_tracker.h5"

    video_path = Path(ROOT_DIR) / f"front_camera.mp4"
    video_ts_path = Path(ROOT_DIR) / f"front_camera_timestamps_{ROBOT_NAME}.csv"
    if not odometry_path.exists():
        raise FileNotFoundError(f"Odometry file not found: {odometry_path}")
    if not camera_info_path.exists():
        raise FileNotFoundError(f"Camera info file not found: {camera_info_path}")
    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")
    if not video_ts_path.exists():
        raise FileNotFoundError(f"Video timestamps file not found: {video_ts_path}")
    if not tf_path.exists():
        raise FileNotFoundError(f"TF static file not found: {tf_path}")
    if not tracker_path.exists():
        raise FileNotFoundError(f"Tracker path file not found: {tracker_path}")
    
    # Load all files
    odom_df = pd.read_csv(odometry_path, header=0)
    odom_np = odom_df.to_numpy()

    # # BEGIN ODOMETRY OVERRIDE
    # override_odom_path = "/home/ec2-user/playground/spinflow/scripts/fai/lhy_dlio_aih_madero_2025-06-12_ferrite5_2025-06-12-15-47-00_test-robot-dlio-pose-odom.csv"
    # odom_df = pd.read_csv(override_odom_path, header=0)

    # # Standardize odometry columns
    # columns = [".header.stamp.secs", ".header.stamp.nsecs", ".pose.pose.position.x", ".pose.pose.position.y", ".pose.pose.position.z", ".pose.pose.orientation.w", ".pose.pose.orientation.x", ".pose.pose.orientation.y", ".pose.pose.orientation.z"]
    # odom_df = odom_df[columns]
    # odom_df['timestamp'] = odom_df['.header.stamp.secs'] + odom_df['.header.stamp.nsecs'] * 1e-9
    # odom_df = odom_df.drop(columns=['.header.stamp.secs', '.header.stamp.nsecs'])
    # odom_df = odom_df[['timestamp', '.pose.pose.position.x', '.pose.pose.position.y', '.pose.pose.position.z', '.pose.pose.orientation.w', '.pose.pose.orientation.x', '.pose.pose.orientation.y', '.pose.pose.orientation.z']]
    # odom_np = odom_df.to_numpy()
    # # END ODOMETRY OVERRIDE


    with open(camera_info_path, 'r') as f:
        camera_info = yaml.safe_load(f)
    with open(tf_path, 'r') as f:
        tf_info = yaml.safe_load(f)

    video_reader = decord.VideoReader(str(video_path))
    video_np = video_reader.get_batch(range(START_FRAME, END_FRAME)).asnumpy()
    video_np = video_np.astype(np.uint8)[:, :, :, [2, 1, 0]]  # Convert RGB to BGR
    video_ts_df = pd.read_csv(video_ts_path, header=0)
    video_ts_np = video_ts_df.to_numpy()[:, -1]

    tracker_info = hkl.load(tracker_path)
    FRONT_RGB = tracker_info['front_rgb'][:, :, [2, 1, 0]]  # Convert RGB to BGR

    # Print data shapes and basic info
    print(f"Odometry data shape: {odom_np.shape}")
    print(f"Camera info: {camera_info}")
    print(f"Video data shape: {video_np.shape}")
    print(f"Video timestamps shape: {video_ts_np.shape}")
    print(f"TF info: {tf_info}")

    video_np_short = video_np # video_np[START_FRAME:END_FRAME]
    video_ts_np_short = video_ts_np[START_FRAME:END_FRAME]

    interp_odo = interpolate_se3(
        cam_ts=video_ts_np_short,
        odo_ts=odom_np[:, 0],
        odo_xyz=odom_np[:, 1:4],
        odo_q_wxyz=odom_np[:, 4:]
    )
    interp_odo_short = interp_odo[:len(video_np_short)]
    interp_odo_short_rel = odom_to_local_pose(interp_odo_short, mode="se3")
    interp_odo_short_ts = video_ts_np_short
    plot_odometry_topdown(interp_odo_short_rel[:, 1:3], "test_plot.png")
    np.set_printoptions(precision=3, suppress=True, linewidth=1000)
    # print(f"XYZ shape: {xyz.shape}, first 5 points:\n{xyz[:len(xyz):10]}")

    # Interpolate odometry to match video frame timestamps
    first_key = list(tf_info.keys())[0]
    print("First key in TF info:", first_key)
    T_optical_to_base = torch.tensor(tf_info[first_key]).view(4, 4).unsqueeze(0)
    infos = {
        "intrinsics": {
            "K": torch.tensor(camera_info['K']).view(3, 3).unsqueeze(0),
            "R": torch.tensor(camera_info['R']).view(3, 3).unsqueeze(0),
            "image_height": torch.tensor([camera_info['image_height']]),
            "image_width": torch.tensor([camera_info['image_width']]),
        },
        "T_optical_to_base": T_optical_to_base
    }
    print("Camera intrinsics:", infos['intrinsics'])
    print("T_optical_to_base:", T_optical_to_base)

    # Save a video with the trajectory drawn on each frame up until the END_FRAME
    base_path_img = draw_trajectory_on_video(
        video_np_short,
        video_ts_np_short,
        interp_odo_short,
        interp_odo_short_ts,
        infos,
        tracker_info=tracker_info,
        enable_cam_offset=False,
        video_path=f"outputs/{ride_path}_base.mp4"
    )
    cam_path_img = draw_trajectory_on_video(
        video_np_short,
        video_ts_np_short,
        interp_odo_short,
        interp_odo_short_ts,
        infos,
        tracker_info=tracker_info,
        enable_cam_offset=True,
        video_path=f"outputs/{ride_path}_cam.mp4"
    )
    # Save the annotated video frames
    output_cam_base_path = Path(f"outputs/{ride_path}.jpg")
    if not output_cam_base_path.parent.exists():
        output_cam_base_path.parent.mkdir(parents=True, exist_ok=True)

    # Draw "base" on the left and "cam" on the right
    base_path_img = cv2.putText(
        base_path_img.copy(), "Base", (10, 30), 
        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA
    )
    cam_path_img = cv2.putText(
        cam_path_img.copy(), "Camera", (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA
    )
    joint_img = np.hstack([base_path_img, cam_path_img])

    cv2.imwrite(str(output_cam_base_path), joint_img)
    print(f"Annotated image frames saved to: {output_cam_base_path}")


Parsed inputs: {'ride_dir': '2025-08-04-16-54-00', 'robot_name': 'ferrite4', 'drive_dir': '2025-08-04-16-58-26_0', 'start_frame': 5577, 'end_frame': 5727}
Odometry data shape: (7755, 8)
Camera info: {'image_width': 640, 'image_height': 480, 'distortion_model': 'plumb_bob', 'frame_id': 'ferrite4/camera_front/camera_color_optical_frame', 'D': [-0.0559946708381176, 0.06892188638448715, -0.00012122321641072631, 0.0006381705752573907, -0.022216971963644028], 'K': [383.0282897949219, 0.0, 327.01702880859375, 0.0, 382.64544677734375, 239.53622436523438, 0.0, 0.0, 1.0], 'R': [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0], 'P': [383.0282897949219, 0.0, 327.01702880859375, 0.0, 0.0, 382.64544677734375, 239.53622436523438, 0.0, 0.0, 0.0, 1.0, 0.0]}
Video data shape: (150, 480, 640, 3)
Video timestamps shape: (11680,)
TF info: {'ferrite4/base_link ferrite4/camera_front/camera_color_optical_frame': [-0.010957589368617154, -0.46306682736297844, 0.8862556316499288, 0.361844, -0.9995874058993858, 0.028